In [0]:
# ============================================================
# BANKING PARTNER COMMISSION & PAYOUT PROJECT
# BRONZE LAYER - SOURCE FILE VERIFICATION
# ============================================================

SOURCE_PATH = "/Volumes/banking_payout/bronze/raw_source_files"

files = dbutils.fs.ls(SOURCE_PATH)

for file in files:
    print(f"{file.name:<40} {file.size:>12,} bytes")

print("-" * 60)
print(f"Total files found: {len(files)}")

src_adjustments.csv                            38,782 bytes
src_casa_acquisitions.csv                     178,207 bytes
src_disbursements.csv                         323,874 bytes
src_incentive_slabs.csv                         5,535 bytes
src_partner_agreements.csv                     11,644 bytes
src_partner_invoices.csv                       20,589 bytes
src_partner_kyc.csv                            12,488 bytes
src_partner_master.csv                         12,864 bytes
src_product_master.csv                            376 bytes
src_rate_cards.csv                             12,497 bytes
src_tax_rules.csv                               1,186 bytes
------------------------------------------------------------
Total files found: 11


In [0]:
# Verify project catalog and schemas

spark.sql("SHOW CATALOGS").show(truncate=False)

spark.sql("SHOW SCHEMAS IN banking_payout").show(truncate=False)

+--------------+
|catalog       |
+--------------+
|banking_payout|
|demo          |
|samples       |
|system        |
|workspace     |
+--------------+

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



In [0]:
from datetime import datetime, timezone
from uuid import uuid4

# Project configuration
CATALOG = "banking_payout"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "/Volumes/banking_payout/bronze/raw_source_files/"

# One batch ID and timestamp shared by all 11 datasets
BATCH_ID = f"BRONZE_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{uuid4().hex[:6]}"
INGESTION_TIMESTAMP = datetime.now(timezone.utc)

# Expected source record counts
EXPECTED_COUNTS = {
    "src_partner_master": 100,
    "src_partner_kyc": 130,
    "src_partner_agreements": 120,
    "src_product_master": 6,
    "src_rate_cards": 100,
    "src_incentive_slabs": 40,
    "src_disbursements": 2000,
    "src_casa_acquisitions": 1000,
    "src_tax_rules": 8,
    "src_adjustments": 150,
    "src_partner_invoices": 100
}

print("Bronze ingestion configuration initialized.")
print(f"Target: {CATALOG}.{BRONZE_SCHEMA}")
print(f"Batch ID: {BATCH_ID}")
print(f"Datasets configured: {len(EXPECTED_COUNTS)}")
print(f"Expected total records: {sum(EXPECTED_COUNTS.values())}")

Bronze ingestion configuration initialized.
Target: banking_payout.bronze
Batch ID: BRONZE_20261127_132630_dd6655
Datasets configured: 11
Expected total records: 3754


In [0]:
from pyspark.sql import functions as F

# Store ingestion results
ingestion_results = []

print("Starting Bronze ingestion...")
print("=" * 70)

for table_name, expected_count in EXPECTED_COUNTS.items():

    file_path = f"{SOURCE_PATH}{table_name}.csv"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    print(f"\nProcessing: {table_name}")

    # Read source CSV without inferring data types
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .load(file_path)
    )

    # Add ingestion metadata
    df = (
        df
        .withColumn("source_file_name", F.lit(f"{table_name}.csv"))
        .withColumn(
            "ingestion_timestamp",
            F.to_timestamp(F.lit(INGESTION_TIMESTAMP.isoformat()))
        )
        .withColumn("batch_id", F.lit(BATCH_ID))
    )

    # Write as a managed Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    # Validate record count
    actual_count = spark.table(target_table).count()

    ingestion_results.append({
        "table_name": table_name,
        "expected_count": expected_count,
        "actual_count": actual_count,
        "status": "PASS" if actual_count == expected_count else "FAIL"
    })

    print(
        f"Expected: {expected_count:>5,} | "
        f"Actual: {actual_count:>5,} | "
        f"{ingestion_results[-1]['status']}"
    )

print("\nBronze ingestion process completed.")

Starting Bronze ingestion...

Processing: src_partner_master
Expected:   100 | Actual:   100 | PASS

Processing: src_partner_kyc
Expected:   130 | Actual:   130 | PASS

Processing: src_partner_agreements
Expected:   120 | Actual:   120 | PASS

Processing: src_product_master
Expected:     6 | Actual:     6 | PASS

Processing: src_rate_cards
Expected:   100 | Actual:   100 | PASS

Processing: src_incentive_slabs
Expected:    40 | Actual:    40 | PASS

Processing: src_disbursements
Expected: 2,000 | Actual: 2,000 | PASS

Processing: src_casa_acquisitions
Expected: 1,000 | Actual: 1,000 | PASS

Processing: src_tax_rules
Expected:     8 | Actual:     8 | PASS

Processing: src_adjustments
Expected:   150 | Actual:   150 | PASS

Processing: src_partner_invoices
Expected:   100 | Actual:   100 | PASS

Bronze ingestion process completed.


In [0]:
from pyspark.sql import functions as F

metadata_columns = [
    "source_file_name",
    "ingestion_timestamp",
    "batch_id"
]

validation_results = []

for table_name in EXPECTED_COUNTS:

    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    df = spark.table(full_table_name)

    # Identify original source columns
    source_columns = [
        field.name for field in df.schema.fields
        if field.name not in metadata_columns
    ]

    # Check original source columns remain strings
    non_string_columns = [
        field.name for field in df.schema.fields
        if field.name not in metadata_columns
        and field.dataType.simpleString() != "string"
    ]

    # Check metadata for nulls
    null_metadata = df.filter(
        F.col("source_file_name").isNull()
        | F.col("ingestion_timestamp").isNull()
        | F.col("batch_id").isNull()
    ).count()

    # Check batch ID and source filename
    batch_ids = [
        row["batch_id"]
        for row in df.select("batch_id").distinct().collect()
    ]

    source_files = [
        row["source_file_name"]
        for row in df.select("source_file_name").distinct().collect()
    ]

    checks_passed = (
        len(non_string_columns) == 0
        and null_metadata == 0
        and batch_ids == [BATCH_ID]
        and source_files == [f"{table_name}.csv"]
    )

    validation_results.append({
        "table": table_name,
        "source_columns": len(source_columns),
        "non_string_columns": len(non_string_columns),
        "null_metadata_rows": null_metadata,
        "batch_id_check": batch_ids == [BATCH_ID],
        "source_file_check": source_files == [f"{table_name}.csv"],
        "status": "PASS" if checks_passed else "FAIL"
    })

# Display validation results
for result in validation_results:
    print(result)

print("\n" + "=" * 70)
print("BRONZE METADATA AND SCHEMA VALIDATION")

passed = sum(r["status"] == "PASS" for r in validation_results)

print(f"Tables validated: {len(validation_results)}")
print(f"Passed: {passed}")
print(f"Failed: {len(validation_results) - passed}")

{'table': 'src_partner_master', 'source_columns': 9, 'non_string_columns': 0, 'null_metadata_rows': 0, 'batch_id_check': True, 'source_file_check': True, 'status': 'PASS'}
{'table': 'src_partner_kyc', 'source_columns': 9, 'non_string_columns': 0, 'null_metadata_rows': 0, 'batch_id_check': True, 'source_file_check': True, 'status': 'PASS'}
{'table': 'src_partner_agreements', 'source_columns': 9, 'non_string_columns': 0, 'null_metadata_rows': 0, 'batch_id_check': True, 'source_file_check': True, 'status': 'PASS'}
{'table': 'src_product_master', 'source_columns': 5, 'non_string_columns': 0, 'null_metadata_rows': 0, 'batch_id_check': True, 'source_file_check': True, 'status': 'PASS'}
{'table': 'src_rate_cards', 'source_columns': 16, 'non_string_columns': 0, 'null_metadata_rows': 0, 'batch_id_check': True, 'source_file_check': True, 'status': 'PASS'}
{'table': 'src_incentive_slabs', 'source_columns': 16, 'non_string_columns': 0, 'null_metadata_rows': 0, 'batch_id_check': True, 'source_file_